# YouTube 姿態辨識 / 行為事件分析 — Demo Notebook

這份 Notebook 是**互動測試用**的簡化版本，方便在 Colab / Jupyter 快速驗證：
1. YOLOv8-Pose 模型能否正常載入與推論
2. 從 YouTube 網址解析串流 URL 是否成功
3. 單張畫面跑完「偵測 → 追蹤 → 行為分析」的完整流程，並印出結果

> 完整的長時間運行 Pipeline（含 Database / Dashboard / Alert）請改用專案根目錄的
> `main.py`（背景執行）+ `streamlit run dashboard/app.py`（視覺化），
> 這兩者不適合直接放在 Notebook 裡跑（會佔用整個 cell 且無法同時開兩個視窗）。


## 1. 安裝相依套件

如果是在 Colab 執行，取消下面這行的註解。

In [ ]:
# !pip install -r ../requirements.txt


## 2. 載入專案模組

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))  # 讓 Notebook 找得到專案根目錄的 src/

from src.config_loader import load_config
from src.stream_capture import extract_stream_url, StreamCapture
from src.pose_detector import PoseDetector
from src.tracker import PersonTracker
from src.behavior_analysis import BehaviorAnalyzer
from src.event_engine import EventEngine
from src.database import Database
from src.alert import AlertDispatcher, generate_ai_report

config = load_config("../config.yaml")
print("YouTube URL:", config.source.youtube_url)


## 3. 測試：解析 YouTube 串流 URL

In [ ]:
stream_url = extract_stream_url(config.source.youtube_url, config.source.preferred_max_height)
print("解析結果:", "成功" if stream_url else "失敗")


## 4. 載入 YOLOv8-Pose 並對單張畫面測試

第一次執行會自動從 ultralytics 官方 GitHub 下載權重檔（`yolov8n-pose.pt`）。


In [ ]:
import cv2
import numpy as np

detector = PoseDetector(
    weights=config.model.weights,
    device=config.model.device,
    conf_threshold=config.model.conf_threshold,
    iou_threshold=config.model.iou_threshold,
    tracker=config.tracking.tracker,
)

# 抓一張畫面來測試：優先用真實串流，若無法取得則用全黑測試畫面（純檢查程式能否跑通）
frame = None
if stream_url:
    cap = cv2.VideoCapture(stream_url)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        frame = None

if frame is None:
    print("無法取得即時畫面，改用全黑測試畫面（僅測試程式流程，不會有偵測結果）")
    frame = np.zeros((720, 1280, 3), dtype=np.uint8)

detections = detector.infer(frame, use_tracking=True)
print(f"偵測到 {len(detections)} 個人")


## 5. 追蹤 + 行為分析（人數統計 / 異常偵測 / ROI）

In [ ]:
tracker = PersonTracker(
    max_history=config.tracking.max_history,
    lost_track_ttl_sec=config.tracking.lost_track_ttl_sec,
)
analyzer = BehaviorAnalyzer(config, tracker)

tracks = tracker.update(detections)
people_stats, events = analyzer.analyze(detections, tracks)

print("人數統計:", people_stats)
print(f"本幀觸發 {len(events)} 筆事件:")
for e in events:
    print(" -", e.event_type, "|", e.severity, "|", e.message)


## 6. 寫入資料庫 + Event Engine（去重 / 分級 / 告警）

In [ ]:
db = Database(path="../data/events.db")
alert_dispatcher = AlertDispatcher(
    console=config.alert.console,
    webhook_url=config.alert.webhook_url,
    min_severity=config.alert.min_severity,
)
event_engine = EventEngine(
    database=db,
    alert_dispatcher=alert_dispatcher,
    severity_map=dict(config.event_engine.severity_map),
    dedupe_window_sec=config.event_engine.dedupe_window_sec,
)

recorded = event_engine.process(events)
print(f"寫入資料庫 {len(recorded)} 筆事件")

for row in db.recent_events(limit=10):
    print(row)

db.close()


## 7. AI 摘要報告

若已設定環境變數 `ANTHROPIC_API_KEY`，會呼叫 Claude 生成自然語言摘要；
否則自動使用內建模板（離線也能用）。


In [ ]:
sample_events = [
    {"severity": "critical", "event_type": "fall", "zone": "入口區", "message": "追蹤 ID 1 疑似跌倒"},
    {"severity": "warning", "event_type": "roi_overcrowd", "zone": "入口區", "message": "區域「入口區」人數 3 超過容量 1"},
]
print(generate_ai_report(sample_events))


## 8. 接下來：跑完整 Pipeline + Dashboard

在終端機（不是 Notebook）執行：

```bash
python main.py --no-display        # 背景持續跑 Pipeline
streamlit run dashboard/app.py      # 另開一個終端機，啟動視覺化 Dashboard
```
